# Amazon Bedrock Agents
Amazon Bedrock Agents is a powerful feature that transforms how developers create AI-powered applications capable of completing complex tasks autonomously.

Whether you're building customer service solutions, internal tools, or data analysis applications, Bedrock Agents handles the complex orchestration between foundation models, data sources, and APIs. By combining advanced language models with seamless integration capabilities, it enables your applications to understand user intent, execute actions, and maintain contextual conversations.

## Key components and capabilities
### Foundation Model integration
At the heart of every Bedrock Agent is a foundation model that powers its understanding and decision-making capabilities. When creating an agent, you'll:

- Select from AWS's collection of foundation models available through Amazon Bedrock
- Configure the model to align with your use case through system prompt configuration
- Let the agent use this model to process user inputs, determine when tools are needed, and orchestrate responses

### Agent instructions
Think of agent instructions as your agent's mission statement and operating manual combined. These instructions define the boundaries and personality of your agent, guiding its interactions and decision-making processes.

For example, an HR assistant agent might be instructed to:
- Verify employee eligibility
- Check available leave balance
- Process vacation requests
- Maintain a professional, helpful tone

### Code interpretation
Code interpretation allows agents to generate and execute code in a secure sandbox environment, enabling:

- Real-time data analysis
- Complex calculations
- Format conversions
- Data visualization
- Custom data processing workflows

### Interactive user inputs
The interactive nature of Bedrock Agents is demonstrated through its sophisticated conversation management. During interactions, agents can:

- Request specific information when needed
- Validate user inputs
- Maintain context throughout the interaction
- Guide users through multi-step processes

### Action groups
Action groups serve as your agent's toolkit for executing tasks. These action groups can include Lambda functions, API integrations, and custom tools. Agents can also connect to knowledge bases to access sources like:

- Company policies
- Product documentation
- Technical guides
- FAQs
- Historical data

### Memory
Bedrock Agents can maintain conversation context through its memory capabilities. Memory enables agents to:

- Retain context across multiple user sessions
- Recall and reference past interactions
- Store summarized conversations using the foundation model
- Configure retention periods by:
    - Number of days
    - Number of sessions
- Access relevant historical information when needed

### Knowledge Base Integration
Bedrock Agents can be associated with one or more knowledge bases to enhance their responses. This integration:

- Enables Retrieval Augmented Generation (RAG)
- Allows agents to access domain-specific information such as:
    - Corporate policies
    - Technical documentation
    - Product information
    - Training materials
- Augments LLM responses with verified information
- Provides real-time access to updated company knowledge
- Helps ensure accurate and consistent responses

## Advanced Capabilities
### Orchestration
Bedrock Agents use orchestration prompts to manage complex tasks and interactions. The orchestration process:

- Combines multiple components to build comprehensive responses:
    - Agent instructions
    - Action group definitions
    - Knowledge base content
- Uses system instructions alongside user chat interactions
- Comes with default prompt templates for common scenarios
- Allows customization through advanced prompts for specific needs
- Manages the flow of:
    - User requests
    - Model interactions
    - Function calls
    - Data retrieval

### Mult-agent collaboration
Bedrock Agents can work together as collaborators to handle complex workflows. This collaboration enables:

- Association of multiple specialized agents
- Reuse of existing agent capabilities, such as:
    - Flight booking agents
    - Calendar management agents
    - Data processing agents
- Orchestration of responses across multiple agents
- Division of complex tasks into specialized functions
- Seamless handoff between different agent capabilities
- Maintenance of context across agent interactions

## Exercise Overview
In this lab, you'll create an agent using Amazon Bedrock that will have access to a tool hosted by AWS Lambda that will be able to access and modify sample employee data.

- ✅ Create an Amazon Bedrock agent
- ✅ Create an AWS Lambda function to access and modify sample employee data
- ✅ Create an action group for the Amazon Bedrock agent
- ✅ Test the agent with different prompts

In [1]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging

print(boto3.__version__)

1.42.13


In [28]:
logger = logging.getLogger(__name__)

# Configurar logging para mostrar no notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()  # Para mostrar no notebook
    ]
)

# Configurar o logger específico
logger.setLevel(logging.INFO)


In [3]:
sts_client = boto3.client('sts')
iam_client = boto3.client('iam')
lambda_client = boto3.client('lambda')
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime')

In [4]:
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]

print(f"Region: {region}, Account ID: {account_id}")

Region: us-east-1, Account ID: 206204551974


In [6]:
inference_profile = "us.amazon.nova-micro-v1:0" 
foundation_model = inference_profile[3:]

print(f"Foundation Model: {foundation_model}")

Foundation Model: amazon.nova-micro-v1:0


In [7]:
suffix = f"{region}-{account_id}"
agent_name = "hr-assistant-function-def"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f'AmazonBedrockExecutionRoleForAgents_{agent_name}'
agent_description = "Agent for providing HR assistance to manage vacation time"
agent_instruction = "You are an HR agent, helping employees understand HR policies and manage vacation time"
agent_action_group_name = "VacationsActionGroup"
agent_action_group_description = "Actions for getting the number of available vacations days for an employee and confirm new time off"
agent_alias_name = f"{agent_name}-alias"
lambda_function_role = f'{agent_name}-lambda-role-{suffix}'
lambda_function_name = f'{agent_name}-{suffix}'

In [8]:
# creating employee database to be used by lambda function
import sqlite3
import random
from datetime import date, timedelta

# Connect to the SQLite database (creates a new one if it doesn't exist)
conn = sqlite3.connect('employee_database.db')
c = conn.cursor()

# Create the employees table
c.execute('''CREATE TABLE IF NOT EXISTS employees
                (employee_id INTEGER PRIMARY KEY AUTOINCREMENT, employee_name TEXT, employee_job_title TEXT, employee_start_date TEXT, employee_employment_status TEXT)''')

# Create the vacations table
c.execute('''CREATE TABLE IF NOT EXISTS vacations
                (employee_id INTEGER, year INTEGER, employee_total_vacation_days INTEGER, employee_vacation_days_taken INTEGER, employee_vacation_days_available INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Create the planned_vacations table
c.execute('''CREATE TABLE IF NOT EXISTS planned_vacations
                (employee_id INTEGER, vacation_start_date TEXT, vacation_end_date TEXT, vacation_days_taken INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Generate some random data for 10 employees
employee_names = ['John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Williams', 'Tom Brown', 'Emily Davis', 'Michael Wilson', 'Sarah Taylor', 'David Anderson', 'Jessica Thompson']
job_titles = ['Manager', 'Developer', 'Designer', 'Analyst', 'Accountant', 'Sales Representative']
employment_statuses = ['Active', 'Inactive']

for i in range(10):
    name = employee_names[i]
    job_title = random.choice(job_titles)
    start_date = date(2015 + random.randint(0, 7), random.randint(1, 12), random.randint(1, 28)).strftime('%Y-%m-%d')
    employment_status = random.choice(employment_statuses)
    c.execute("INSERT INTO employees (employee_name, employee_job_title, employee_start_date, employee_employment_status) VALUES (?, ?, ?, ?)", (name, job_title, start_date, employment_status))
    employee_id = c.lastrowid

    # Generate vacation data for the current employee
    for year in range(date.today().year, date.today().year - 3, -1):
        total_vacation_days = random.randint(10, 30)
        days_taken = random.randint(0, total_vacation_days)
        days_available = total_vacation_days - days_taken
        c.execute("INSERT INTO vacations (employee_id, year, employee_total_vacation_days, employee_vacation_days_taken, employee_vacation_days_available) VALUES (?, ?, ?, ?, ?)", (employee_id, year, total_vacation_days, days_taken, days_available))

        # Generate some planned vacations for the current employee and year
        num_planned_vacations = random.randint(0, 3)
        for _ in range(num_planned_vacations):
            start_date = date(year, random.randint(1, 12), random.randint(1, 28)).strftime('%Y-%m-%d')
            end_date = (date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])) + timedelta(days=random.randint(1, 14))).strftime('%Y-%m-%d')
            days_taken = (date(int(end_date[:4]), int(end_date[5:7]), int(end_date[8:])) - date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])))
            c.execute("INSERT INTO planned_vacations (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken) VALUES (?, ?, ?, ?)", (employee_id, start_date, end_date, days_taken.days))

# Commit the changes and close the connection
conn.commit()
conn.close()


In [ ]:
%%writefile lambda_function.py
import os
import json
import sqlite3
from datetime import datetime

def get_available_vacations_days(employee_id):
    
    conn = sqlite3.connect('employee_database.db')
    c = conn.cursor()

    if not employee_id:
        conn.close()
        raise ValueError("Employee ID is required")

    c.execute("""
        SELECT employee_vacation_days_available
        FROM vacations
        WHERE employee_id = ?
        ORDER BY year DESC
        LIMIT 1
    """, (str(employee_id),))

    result = c.fetchone()
    conn.close()

    if result:
        return result[0]
    else:
        return f"No vacation data found for employee_id {employee_id}"

def lambda_handler(event, context):
    try:
        print(f"Received event: {json.dumps(event)}")
        
        # Bedrock Agent passa parâmetros em diferentes formatos
        employee_id = None
        
        # Tentar diferentes formatos de entrada
        if 'employee_id' in event:
            employee_id = event['employee_id']
        elif 'parameters' in event:
            for param in event['parameters']:
                if param['name'] == 'employee_id':
                    employee_id = param['value']
                    break
        
        if not employee_id:
            raise ValueError("employee_id parameter is required")
        
        available_days = get_available_vacations_days(employee_id)
        
        # Formato de resposta para Bedrock Agent
        response = {
            'response': {
                'actionGroup': event.get('actionGroup', 'VacationsActionGroup'),
                'function': event.get('function', 'get_available_vacations_days'),
                'functionResponse': {
                    'responseBody': {
                        'TEXT': {
                            'body': json.dumps({
                                'employee_id': employee_id,
                                'available_vacation_days': available_days
                            })
                        }
                    }
                }
            }
        }
        
        print(f"Returning response: {json.dumps(response)}")
        return response
        
    except Exception as e:
        print(f"Error: {str(e)}")
        
        # Formato de erro para Bedrock Agent
        error_response = {
            'response': {
                'actionGroup': event.get('actionGroup', 'VacationsActionGroup'),
                'function': event.get('function', 'get_available_vacations_days'),
                'functionResponse': {
                    'responseBody': {
                        'TEXT': {
                            'body': json.dumps({
                                'error': str(e)
                            })
                        }
                    }
                }
            }
        }
        
        return error_response

Writing lambda_function.py


In [10]:
# Create the Lambda IAM role and policy to invoke a Bedrock model
try:
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "lambda.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    assume_role_policy_document_json = json.dumps(assume_role_policy_document)

    lambda_iam_role = iam_client.create_role(
        RoleName=lambda_function_role,
        AssumeRolePolicyDocument=assume_role_policy_document_json
    )

    # Pause to make sure role is created
    time.sleep(10)
except:
    lambda_iam_role = iam_client.get_role(RoleName=lambda_function_role)

iam_client.attach_role_policy(
    RoleName=lambda_function_role,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

{'ResponseMetadata': {'RequestId': 'f2ca4a9b-28aa-4061-af3b-41116d4b9d9d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:03:32 GMT',
   'x-amzn-requestid': 'f2ca4a9b-28aa-4061-af3b-41116d4b9d9d',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

Package the Lambda function to a Zip file and create the Lambda function using boto3

In [ ]:
# Package up the lambda function code
s = BytesIO()
z = zipfile.ZipFile(s, 'w')
z.write("lambda_function.py")
z.write("employee_database.db")
z.close()
zip_content = s.getvalue()

# Create Lambda Function
lambda_function = lambda_client.create_function(
    FunctionName=lambda_function_name,
    Runtime='python3.14', 
    Timeout=180,
    Role=lambda_iam_role['Role']['Arn'],
    Code={'ZipFile': zip_content},
    Handler='lambda_function.lambda_handler'
)

We need to give it the right permissions:

- 🔐 Create the agent policies that allow it to invoke Bedrock foundation models
- 🛡️ Create the IAM role for the agent and attach the policies

These steps will allow the agent to safely and successfully use the tools.

In [12]:
# We will create the IAM policy that the agent will need to use the inference profile and invoke the model. 
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                f"arn:aws:bedrock:*::foundation-model/{foundation_model}",
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        },
        {
            "Sid": "AmazonBedrockAgentBedrockGetInferenceProfile",
            "Effect": "Allow",
            "Action":  [
                "bedrock:GetInferenceProfile",
                "bedrock:ListInferenceProfiles",
                "bedrock:UseInferenceProfile"
            ],
            "Resource": [
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        }
    ]
}

bedrock_policy_json = json.dumps(bedrock_agent_bedrock_allow_policy_statement)

agent_bedrock_policy = iam_client.create_policy(
    PolicyName=agent_bedrock_allow_policy_name,
    PolicyDocument=bedrock_policy_json
)

Now, we need to create the IAM role that the agent will need to assume to perform its duties. We will also attach an IAM policy to the role

In [13]:
# Create IAM Role for the agent and attach IAM policies
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "bedrock.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}

assume_role_policy_document_json = json.dumps(assume_role_policy_document)
agent_role = iam_client.create_role(
    RoleName=agent_role_name,
    AssumeRolePolicyDocument=assume_role_policy_document_json
)

# Pause to make sure role is created
time.sleep(10)

iam_client.attach_role_policy(
    RoleName=agent_role_name,
    PolicyArn=agent_bedrock_policy['Policy']['Arn']
)

{'ResponseMetadata': {'RequestId': '6ed4110d-2634-48ea-add8-121f34c60279',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:09:17 GMT',
   'x-amzn-requestid': '6ed4110d-2634-48ea-add8-121f34c60279',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [ ]:
# Create the Bedrock Agent
response = bedrock_agent_client.create_agent(
    agentName=agent_name,
    agentResourceRoleArn=agent_role['Role']['Arn'],
    description=agent_description,
    idleSessionTTLInSeconds=1800,
    foundationModel=inference_profile,
    instruction=agent_instruction, 
)
agent_id = response['agent']['agentId']
agent_id, response

('JGCZ6RIKSS',
 {'ResponseMetadata': {'RequestId': 'fb83ff0b-eedb-438b-b898-5192d223348b',
   'HTTPStatusCode': 202,
   'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:10:16 GMT',
    'content-type': 'application/json',
    'content-length': '692',
    'connection': 'keep-alive',
    'x-amzn-requestid': 'fb83ff0b-eedb-438b-b898-5192d223348b',
    'x-amz-apigw-id': 'WayyRFRTIAMEO5Q=',
    'x-amzn-trace-id': 'Root=1-695431a7-476491543af41a2471729886'},
   'RetryAttempts': 0},
  'agent': {'agentId': 'JGCZ6RIKSS',
   'agentName': 'hr-assistant-function-def',
   'agentArn': 'arn:aws:bedrock:us-east-1:206204551974:agent/JGCZ6RIKSS',
   'instruction': 'You are an HR agent, helping employees understand HR policies and manage vacation time',
   'agentStatus': 'CREATING',
   'foundationModel': 'us.amazon.nova-micro-v1:0',
   'description': 'Agent for providing HR assistance to manage vacation time',
   'orchestrationType': 'DEFAULT',
   'idleSessionTTLInSeconds': 1800,
   'agentResourceRoleArn': 'a

In [ ]:
# Define the functions that the agent can use
agent_functions = [
    {
        'name': 'get_available_vacations_days',
        'description': 'get the number of vacations available for a certain employee',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee to get the available vacations",
                "required": True,
                "type": "integer"
            }
        }
    },
    {
        'name': 'reserve_vacation_time',
        'description': 'reserve vacation time for a specific employee - you need all parameters to reserve vacation time',
        'parameters': {
            "employee_id": {
                "description": "the id of the employee for which time off will be reserved",
                "required": True,
                "type": "integer"
            },
            "start_date": {
                "description": "the start date for the vacation time",
                "required": True,
                "type": "string" 
            },
            "end_date": {
                "description": "the end date for the vacation time",
                "required": True,
                "type": "string"
            }
        }
    },
]

In [16]:
# Now, we can configure and create an action group here:
agent_action_group_response = bedrock_agent_client.create_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupExecutor={
        'lambda': lambda_function['FunctionArn']
    },
    actionGroupName=agent_action_group_name,
    functionSchema={
        'functions': agent_functions
    },
    description=agent_action_group_description
)

agent_action_group_response

{'ResponseMetadata': {'RequestId': 'f9f0dd68-f3e1-4ab7-b892-842e27b0b4a9',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:12:02 GMT',
   'content-type': 'application/json',
   'content-length': '1335',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'f9f0dd68-f3e1-4ab7-b892-842e27b0b4a9',
   'x-amz-apigw-id': 'WazC3GTNoAMEm9g=',
   'x-amzn-trace-id': 'Root=1-69543211-0083349219999c3c048694a8'},
  'RetryAttempts': 0},
 'agentActionGroup': {'agentId': 'JGCZ6RIKSS',
  'agentVersion': 'DRAFT',
  'actionGroupId': 'UOJ2OFIZVC',
  'actionGroupName': 'VacationsActionGroup',
  'description': 'Actions for getting the number of available vacations days for an employee and confirm new time off',
  'createdAt': datetime.datetime(2025, 12, 30, 20, 12, 2, 52849, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2025, 12, 30, 20, 12, 2, 52849, tzinfo=tzutc()),
  'actionGroupExecutor': {'lambda': 'arn:aws:lambda:us-east-1:206204551974:function:hr-assistant-function-def-us

In [17]:
# Modify the resource policy on the AWS Lambda function to allow the specific bedrock agent to invoke it.
response = lambda_client.add_permission(
    FunctionName=lambda_function_name,
    StatementId='allow_bedrock',
    Action='lambda:InvokeFunction',
    Principal='bedrock.amazonaws.com',
    SourceArn=f"arn:aws:bedrock:{region}:{account_id}:agent/{agent_id}",
)

In [18]:
# Prepare the agent to make it ACTIVE
response = bedrock_agent_client.prepare_agent(
    agentId=agent_id
)
response

{'ResponseMetadata': {'RequestId': 'f6cc6e60-abc2-43a1-91f7-4bb983979371',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:14:13 GMT',
   'content-type': 'application/json',
   'content-length': '119',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'f6cc6e60-abc2-43a1-91f7-4bb983979371',
   'x-amz-apigw-id': 'WazXVEHYIAMEo0A=',
   'x-amzn-trace-id': 'Root=1-69543294-7f35f194316012ef15b48b5b'},
  'RetryAttempts': 0},
 'agentId': 'JGCZ6RIKSS',
 'agentStatus': 'PREPARING',
 'agentVersion': 'DRAFT',
 'preparedAt': datetime.datetime(2025, 12, 30, 20, 14, 13, 192212, tzinfo=tzutc())}

In [19]:
created_agent_id = response['agentId']
print(f"Created agent with ID: {created_agent_id}")

Created agent with ID: JGCZ6RIKSS


Now we will create a alias for our agent by passing it's id and the desired alias

In [ ]:
response = bedrock_agent_client.create_agent_alias(
    agentAliasName='test-alias-1',
    agentId=created_agent_id
)

response

{'ResponseMetadata': {'RequestId': 'd364f7ee-7457-46f1-8560-587ecae72132',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 20:16:16 GMT',
   'content-type': 'application/json',
   'content-length': '382',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'd364f7ee-7457-46f1-8560-587ecae72132',
   'x-amz-apigw-id': 'WazqnF-MIAMEirg=',
   'x-amzn-trace-id': 'Root=1-69543310-6bdd04cd2ff645210769671d'},
  'RetryAttempts': 0},
 'agentAlias': {'agentId': 'JGCZ6RIKSS',
  'agentAliasId': '3OUE1J1SMQ',
  'agentAliasName': 'test-alias-1',
  'agentAliasArn': 'arn:aws:bedrock:us-east-1:206204551974:agent-alias/JGCZ6RIKSS/3OUE1J1SMQ',
  'routingConfiguration': [{}],
  'createdAt': datetime.datetime(2025, 12, 30, 20, 16, 16, 561847, tzinfo=tzutc()),
  'updatedAt': datetime.datetime(2025, 12, 30, 20, 16, 16, 561847, tzinfo=tzutc()),
  'agentAliasStatus': 'CREATING',
  'aliasInvocationState': 'ACCEPT_INVOCATIONS'}}

In [21]:
agent_alias_id = response['agentAlias']['agentAliasId']
print(f"Created agent alias with ID: {agent_alias_id}")

Created agent alias with ID: 3OUE1J1SMQ


Now that we've created the agent, let's use the bedrock-agent-runtime client and the invoke_agent API to invoke this agent and perform some tasks.

In [33]:
## create a random id for session initiator id
session_id: str = str(uuid.uuid1())
enable_trace: bool = True
end_session: bool = False

# invoke the agent API
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="How much vacation does the employee with employee_id set to 1 have available?",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession=end_session
)

logger.info("Agent Response received")
#logger.info(pprint.pformat(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer -> {data.decode('utf8')}")
            agent_answer = data.decode('utf8')
        elif 'trace' in event:
            #logger.info("Trace event:")
            #logger.info(pprint.pformat(event['trace']))
            pass
        else:
            logger.warning(f"Unexpected event: {event}")
except Exception as e:
    logger.error(f"Error processing events: {e}")


2025-12-30 20:11:28,608 - __main__ - INFO - Agent Response received
2025-12-30 20:11:31,002 - __main__ - INFO - Final answer -> The employee with employee_id 1 has 11 vacation days available.


In [34]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="Great. please reserve one day of time off for the employee with employee_id set to 1 for 2026-06-01",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
            # End event indicates that the request finished successfully
        elif 'trace' in event:
            #logger.info(pprint.pformat(event['trace']))
            pass
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

2025-12-30 20:17:58,363 - __main__ - INFO - None


{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/vnd.amazon.eventstream',
                                      'date': 'Tue, 30 Dec 2025 23:17:58 GMT',
                                      'transfer-encoding': 'chunked',
                                      'x-amz-bedrock-agent-session-id': 'deecb6bc-e5d4-11f0-90af-acde48001122',
                                      'x-amzn-bedrock-agent-content-type': 'application/json',
                                      'x-amzn-requestid': 'f6325d5c-211d-4c38-a489-eb8bac8e6ba5'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'f6325d5c-211d-4c38-a489-eb8bac8e6ba5',
                      'RetryAttempts': 0},
 'completion': <botocore.eventstream.EventStream object at 0x10e35b110>,
 'contentType': 'application/json',
 'sessionId': 'deecb6bc-e5d4-11f0-90af-acde48001122'}


2025-12-30 20:18:01,469 - __main__ - INFO - Final answer ->
One day of vacation time has been reserved for the employee with employee_id 1 starting from 2026-06-01.


In [35]:
agentResponse = bedrock_agent_runtime_client.invoke_agent(
    inputText="How much vacation does the employee with employee_id set to 1 have available?",
    agentId=agent_id,
    agentAliasId=agent_alias_id, 
    sessionId=session_id,
    enableTrace=enable_trace, 
    endSession= end_session
)

logger.info(pprint.pprint(agentResponse))

event_stream = agentResponse['completion']
try:
    for event in event_stream:        
        if 'chunk' in event:
            data = event['chunk']['bytes']
            logger.info(f"Final answer ->\n{data.decode('utf8')}")
            agent_answer = data.decode('utf8')
            end_event_received = True
        elif 'trace' in event:
            #logger.info(json.dumps(event['trace'], indent=2))
            pass
        else:
            raise Exception("unexpected event.", event)
except Exception as e:
    raise Exception("unexpected event.", e)

2025-12-30 20:39:49,963 - __main__ - INFO - None


{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/vnd.amazon.eventstream',
                                      'date': 'Tue, 30 Dec 2025 23:39:49 GMT',
                                      'transfer-encoding': 'chunked',
                                      'x-amz-bedrock-agent-session-id': 'deecb6bc-e5d4-11f0-90af-acde48001122',
                                      'x-amzn-bedrock-agent-content-type': 'application/json',
                                      'x-amzn-requestid': '51f1620b-b8ca-43aa-9d5c-0e1ed7c40b62'},
                      'HTTPStatusCode': 200,
                      'RequestId': '51f1620b-b8ca-43aa-9d5c-0e1ed7c40b62',
                      'RetryAttempts': 0},
 'completion': <botocore.eventstream.EventStream object at 0x10e18b830>,
 'contentType': 'application/json',
 'sessionId': 'deecb6bc-e5d4-11f0-90af-acde48001122'}


2025-12-30 20:39:52,774 - __main__ - INFO - Final answer ->
The employee with employee_id 1 now has 10 vacation days available.


## Cleanup

The next steps demonstrate how to delete the agent and associated resources. 

- ❌ Update the action group to disable it
- ❌ Delete agent action group
- ❌ Delete agent
- ❌ Delete lambda function
- ❌ Delete the created IAM roles and policies

In [36]:
action_group_id = agent_action_group_response['agentActionGroup']['actionGroupId']
action_group_name = agent_action_group_response['agentActionGroup']['actionGroupName']

response = bedrock_agent_client.update_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id,
    actionGroupName=action_group_name,
    actionGroupExecutor={
        'lambda': lambda_function['FunctionArn']
    },
    functionSchema={
        'functions': agent_functions
    },
    actionGroupState='DISABLED',
)

action_group_deletion = bedrock_agent_client.delete_agent_action_group(
    agentId=agent_id,
    agentVersion='DRAFT',
    actionGroupId= action_group_id
)

In [37]:
response = bedrock_agent_client.delete_agent_alias(
    agentAliasId=agent_alias_id,
    agentId=agent_id
)

In [38]:
agent_deletion = bedrock_agent_client.delete_agent(
    agentId=agent_id
)

In [39]:
lambda_client.delete_function(
    FunctionName=lambda_function_name
)

{'ResponseMetadata': {'RequestId': '0d0d471b-c952-464d-9992-c6b4b7d6931c',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'date': 'Tue, 30 Dec 2025 23:47:34 GMT',
   'content-type': 'application/json',
   'connection': 'keep-alive',
   'x-amzn-requestid': '0d0d471b-c952-464d-9992-c6b4b7d6931c'},
  'RetryAttempts': 0},
 'StatusCode': 204}

In [41]:
for policy in [agent_bedrock_allow_policy_name]:
    iam_client.detach_role_policy(RoleName=agent_role_name, PolicyArn=f'arn:aws:iam::{account_id}:policy/{policy}')
    
iam_client.detach_role_policy(RoleName=lambda_function_role, PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')

for role_name in [agent_role_name, lambda_function_role]:
    iam_client.delete_role(
        RoleName=role_name
    )

for policy in [agent_bedrock_policy]:
    iam_client.delete_policy(
        PolicyArn=policy['Policy']['Arn']
    )